# 01 - Guida al repository

Questo notebook serve a orientarsi nel repository.

Obiettivi:
- capire quali domande il progetto vuole coprire;
- vedere quali fonti e dataset logici sono gia' definiti;
- controllare quali tabelle finali esistono e quali sono ancora vuote;
- offrire un punto di partenza a utenti che non vogliono leggere subito gli script.

## Parametri modificabili

Questa cella definisce i percorsi principali. Se il notebook viene eseguito dalla cartella `notebooks/`, il repository viene individuato come cartella padre.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CODE_DIR = REPO_ROOT / 'code'
METADATA_DIR = REPO_ROOT / 'metadata'
DATA_FINAL_DIR = REPO_ROOT / 'data' / 'final'

if str(CODE_DIR) not in sys.path:
    sys.path.append(str(CODE_DIR))

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

print(f'Repository: {REPO_ROOT}')

## Funzioni di utilita'

Queste funzioni leggono le tabelle se esistono e restituiscono tabelle vuote se i dati non sono ancora stati prodotti. Non modificano file.

In [ ]:
def read_csv_optional(path: Path) -> pd.DataFrame:
    """Legge un CSV se esiste. Restituisce una tabella vuota se manca."""
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_csv(path)


def show_table(df: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    """Mostra le prime righe in modo sicuro anche se la tabella e' vuota."""
    if df.empty:
        print('Tabella vuota o file non disponibile.')
        return df
    return df.head(n)


def row_count(path: Path) -> int:
    """Conta le righe dati di un CSV, escludendo l intestazione."""
    if not path.exists() or path.stat().st_size == 0:
        return 0
    return max(sum(1 for _ in path.open('r', encoding='utf-8')) - 1, 0)

## Fonti registrate

`registro_fonti.csv` contiene le fonti ufficiali o istituzionali previste dal progetto. Non tutti i dataset tecnici sono gia' collegati alle API, ma il perimetro logico e' dichiarato.

In [ ]:
fonti = read_csv_optional(METADATA_DIR / 'registro_fonti.csv')
show_table(fonti, 20)

## Dataset logici attesi

`dataset_attesi.csv` risponde alla domanda: sappiamo quali fonti servono? Si'. La tabella elenca i dataset logici necessari e li collega a indicatori e tabelle finali.

La distinzione e' questa:
- dataset logico: cosa serve all analisi;
- endpoint tecnico: ID o URL esatto da usare per scaricare automaticamente il dato.

In [ ]:
dataset_attesi = read_csv_optional(METADATA_DIR / 'dataset_attesi.csv')
show_table(dataset_attesi, 30)

## Domande delle live e copertura

`domande_live.csv` collega ogni domanda a un indicatore e a una tabella finale. Serve a evitare che il repository diventi una raccolta di grafici scollegati.

In [ ]:
domande_live = read_csv_optional(METADATA_DIR / 'domande_live.csv')
show_table(domande_live, 35)

## Tabelle finali

Le tabelle finali sono l interfaccia stabile per analisi, grafici e dashboard. Se sono vuote, il progetto e' ancora nella fase di collegamento fonti e trasformazione.

In [ ]:
final_tables = sorted(DATA_FINAL_DIR.glob('*.csv')) if DATA_FINAL_DIR.exists() else []
summary = pd.DataFrame({
    'tabella': [path.name for path in final_tables],
    'percorso': [str(path.relative_to(REPO_ROOT)) for path in final_tables],
    'righe': [row_count(path) for path in final_tables],
})

if summary.empty:
    print('Nessuna tabella finale trovata. Esegui prima: python code/esegui_pipeline.py')
else:
    display(summary)

## Controlli qualita'

Il blocco seguente esegue i controlli definiti nel repository. Il risultato aiuta a capire cosa manca prima di fare analisi numeriche.

In [ ]:
try:
    from controlli_qualita import esegui_controlli_qualita

    log_qualita = esegui_controlli_qualita()
    display(log_qualita)
except Exception as error:
    print('Controlli qualita non eseguiti.')
    print(type(error).__name__, error)

## Lettura dello stato del progetto

Se le tabelle finali sono vuote, il repository non e' rotto. Significa che i dataset logici sono definiti, ma devono essere collegati agli endpoint tecnici e poi trasformati.

Questo notebook resta utile anche dopo il popolamento dei dati: mostrera' automaticamente le righe disponibili e i controlli aggiornati.